# Notebook 2 - Analyse et figures

Dans ce notebook, nous rechargeons les tables finales, construisons des indicateurs utiles et génèrons toutes les figures pour la présentation.
        


In [137]:
from pathlib import Path
import json
import math
import textwrap
import unicodedata
from typing import Dict, Iterable, List

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon
import numpy as np
import pandas as pd
import requests
from shapely.geometry import shape as shapely_shape

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
FIGURES_DIR = PROJECT_DIR / "figures"

MERGED_PATH = PROCESSED_DIR / "ter_weather_current_regions_monthly.csv"
MODE_COMPARISON_PATH = PROCESSED_DIR / "rail_mode_comparison_monthly.csv"
REGION_CONTEXT_PATH = PROCESSED_DIR / "rail_region_context.csv"
REGIONS_GEOJSON_PATH = RAW_DIR / "regions_2025_1000m.geojson"
REGIONS_GEOJSON_URL = "https://object.data.gouv.fr/contours-administratifs/2025/geojson/regions-1000m.geojson"

DISPLAY_REGION_NAMES = {
    "Auvergne-Rhone-Alpes": "Auvergne-Rhône-Alpes",
    "Bourgogne-Franche-Comte": "Bourgogne-Franche-Comté",
    "Bretagne": "Bretagne",
    "Centre-Val de Loire": "Centre-Val de Loire",
    "Grand Est": "Grand Est",
    "Hauts-de-France": "Hauts-de-France",
    "Ile-de-France": "Île-de-France",
    "Normandie": "Normandie",
    "Nouvelle-Aquitaine": "Nouvelle-Aquitaine",
    "Occitanie": "Occitanie",
    "Pays de la Loire": "Pays de la Loire",
    "Provence-Alpes-Cote d'Azur": "Provence-Alpes-Côte d'Azur",
}

DISPLAY_MODE_NAMES = {
    "TER": "TER",
    "Transilien": "Transilien",
    "Intercites": "Intercités",
}

MAP_LABELS = {
    "Auvergne-Rhone-Alpes": "AURA",
    "Bourgogne-Franche-Comte": "BFC",
    "Bretagne": "Bret.",
    "Centre-Val de Loire": "CVL",
    "Grand Est": "GE",
    "Hauts-de-France": "HdF",
    "Ile-de-France": "IDF",
    "Normandie": "Norm.",
    "Nouvelle-Aquitaine": "NAq",
    "Occitanie": "Occ.",
    "Pays de la Loire": "PDL",
    "Provence-Alpes-Cote d'Azur": "PACA",
}

MONTH_ORDER = list(range(1, 13))
MONTH_LABELS = {
    1: "Jan",
    2: "Fev",
    3: "Mar",
    4: "Avr",
    5: "Mai",
    6: "Jun",
    7: "Jul",
    8: "Aou",
    9: "Sep",
    10: "Oct",
    11: "Nov",
    12: "Dec",
}
STRESS_BUCKET_ORDER = ["Mois calmes", "Mois intermediaires", "Mois chocs meteo"]

COLORS = {
    "regularity": "#0B4F6C",
    "cancellation": "#E36414",
    "stress": "#9A031E",
    "accent": "#5F0F40",
    "calm": "#90BE6D",
    "middle": "#F9C74F",
    "shock": "#F94144",
    "grid": "#D0D7DE",
    "text": "#172B4D",
    "ter": "#0B4F6C",
    "intercites": "#E36414",
    "transilien": "#2A9D8F",
}

MODE_COLORS = {
    "TER": COLORS["ter"],
    "Intercites": COLORS["intercites"],
    "Transilien": COLORS["transilien"],
}


def normalize_text(value: object) -> str:
    text = unicodedata.normalize("NFKD", str(value or ""))
    ascii_text = text.encode("ascii", "ignore").decode("ascii")
    ascii_text = ascii_text.replace("-", " ").replace("'", " ").replace("/", " ").replace("?", " ")
    return " ".join(ascii_text.split()).lower()


CANONICAL_REGION_BY_NORMALIZED = {
    normalize_text(display_name): key for key, display_name in DISPLAY_REGION_NAMES.items()
}
CANONICAL_REGION_BY_NORMALIZED.update({normalize_text(key): key for key in DISPLAY_REGION_NAMES})


def canonical_region(value: object) -> str:
    raw = str(value or "").strip()
    return CANONICAL_REGION_BY_NORMALIZED.get(normalize_text(raw), raw)


def display_region(value: object) -> str:
    key = canonical_region(value)
    return DISPLAY_REGION_NAMES.get(key, str(value))


def canonical_mode(value: object) -> str:
    raw = str(value or "").strip()
    normalized = normalize_text(raw)
    if normalized.startswith("intercit"):
        return "Intercites"
    if normalized == "ter":
        return "TER"
    if normalized == "transilien":
        return "Transilien"
    return raw


def display_mode(value: object) -> str:
    key = canonical_mode(value)
    return DISPLAY_MODE_NAMES.get(key, str(value))


def weighted_average(values: pd.Series, weights: pd.Series) -> float:
    mask = values.notna() & weights.notna()
    if not mask.any():
        return math.nan
    return float(np.average(values[mask], weights=weights[mask]))


def linear_fit(x: Iterable[float], y: Iterable[float]) -> dict:
    x_array = np.asarray(list(x), dtype=float)
    y_array = np.asarray(list(y), dtype=float)
    mask = np.isfinite(x_array) & np.isfinite(y_array)
    if mask.sum() < 2:
        return {"slope": math.nan, "intercept": math.nan, "correlation": math.nan}
    slope, intercept = np.polyfit(x_array[mask], y_array[mask], 1)
    correlation = float(np.corrcoef(x_array[mask], y_array[mask])[0, 1])
    return {"slope": float(slope), "intercept": float(intercept), "correlation": correlation}


def shorten_comment(text: object, width: int = 110) -> str:
    if pd.isna(text) or not str(text).strip():
        return "Pas de commentaire SNCF disponible."
    clean = " ".join(str(text).split())
    return textwrap.shorten(clean, width=width, placeholder="...")


def add_value_labels(ax: plt.Axes, padding: float = 0.12, fmt: str = "{:.2f}") -> None:
    for patch in ax.patches:
        value = patch.get_height()
        ax.text(
            patch.get_x() + patch.get_width() / 2,
            value + padding,
            fmt.format(value),
            ha="center",
            va="bottom",
            fontsize=10,
            color=COLORS["text"],
        )


def add_horizontal_labels(ax: plt.Axes, fmt: str = "{:.2f}", offset: float = 0.08) -> None:
    for patch in ax.patches:
        value = patch.get_width()
        ax.text(
            value + offset,
            patch.get_y() + patch.get_height() / 2,
            fmt.format(value),
            ha="left",
            va="center",
            fontsize=9,
            color=COLORS["text"],
        )


def add_source(fig: plt.Figure, text: str) -> None:
    fig.text(0.01, 0.01, text, fontsize=9, color="#5E6C84")


def save_figure(fig: plt.Figure, path: Path) -> None:
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.close(fig)
        


In [138]:
for required_path in [MERGED_PATH, MODE_COMPARISON_PATH, REGION_CONTEXT_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Le fichier {required_path.name} est absent. Lance d'abord le notebook 1."
        )

merged = pd.read_csv(MERGED_PATH, parse_dates=["date"])
mode_monthly = pd.read_csv(MODE_COMPARISON_PATH, parse_dates=["date"])
region_context = pd.read_csv(REGION_CONTEXT_PATH)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

merged["region_current"] = merged["region_current"].map(canonical_region)
mode_monthly["mode"] = mode_monthly["mode"].map(canonical_mode)
region_context["region_current"] = region_context["region_current"].map(canonical_region)

if "stress_bucket" in merged.columns:
    merged["stress_bucket"] = merged["stress_bucket"].replace(
        {
            "Mois intermédiaires": "Mois intermediaires",
            "Mois chocs météo": "Mois chocs meteo",
        }
    )

print("Base TER + meteo :", merged.shape)
print("Base comparaison des modes :", mode_monthly.shape)
print("Base contexte :", region_context.shape)
merged.head()
        


Base TER + meteo : (1542, 47)
Base comparaison des modes : (450, 18)
Base contexte : (12, 7)


,date,region_current,source_regions,nombre_de_trains_programmes,nombre_de_trains_ayant_circule,nombre_de_trains_annules,nombre_de_trains_en_retard_a_l_arrivee,commentaires,regularity_pct,cancellation_pct,...,heat_days_stress,frost_days_z,frost_days_stress,storm_days_z,storm_days_stress,weather_stress_score,dominant_weather_key,dominant_weather_label,stress_bucket,comment_clean
0,2013-01-01,Auvergne-Rhone-Alpes,"Auvergne, Rhône Alpes",37223.0,36511.0,712.0,3983.0,Conditions météos défavorables.,89.090959,1.912796,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Conditions météos défavorables.
1,2013-01-01,Bourgogne-Franche-Comte,"Bourgogne, Franche Comté",14226.0,14076.0,150.0,1178.0,Un mois de janvier qui surpasse les six exerci...,91.631145,1.054407,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Un mois de janvier qui surpasse les six exerci...
2,2013-01-01,Bretagne,Bretagne,8776.0,8631.0,145.0,554.0,Fortes chutes de neige ayant entrainé des pert...,93.581277,1.652233,...,0.0,0.404067,0.404067,0.0,0.0,0.339054,snow_days,Neige,Mois intermediaires,Fortes chutes de neige ayant entrainé des pert...
3,2013-01-01,Centre-Val de Loire,Centre,9882.0,9687.0,195.0,812.0,"Trois incidents caténaires lourds, dont deux p...",91.617632,1.973285,...,0.0,0.550090,0.550090,0.0,0.0,0.412076,snow_days,Neige,Mois intermediaires,"Trois incidents caténaires lourds, dont deux p..."
4,2013-01-01,Grand Est,"Alsace, Champagne Ardenne, Lorraine",NaN,NaN,NaN,NaN,Conditions météorologiques dégradées du 15 au ...,NaN,NaN,...,0.0,0.438215,0.438215,0.0,0.0,0.155910,snow_days,Neige,Mois intermediaires,Conditions météorologiques dégradées du 15 au ...


## Indicateurs
        


In [139]:
def build_national_monthly(merged: pd.DataFrame) -> pd.DataFrame:
    rows: List[dict] = []
    for date, group in merged.groupby("date", sort=True):
        programmed = group["nombre_de_trains_programmes"].sum()
        circulated = group["nombre_de_trains_ayant_circule"].sum()
        cancelled = group["nombre_de_trains_annules"].sum()
        delayed = group["nombre_de_trains_en_retard_a_l_arrivee"].sum()
        rows.append(
            {
                "date": date,
                "regularity_pct": 100 * (circulated - delayed) / circulated,
                "cancellation_pct": 100 * cancelled / programmed,
                "weather_stress_score": weighted_average(group["weather_stress_score"], group["nombre_de_trains_programmes"]),
                "regions_covered": int(group["region_current"].nunique()),
            }
        )
    national = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
    national["regularity_rolling_12m"] = national["regularity_pct"].rolling(12, min_periods=3).mean()
    national["stress_rolling_12m"] = national["weather_stress_score"].rolling(12, min_periods=3).mean()
    return national


def build_month_profiles(merged: pd.DataFrame) -> dict:
    month_regularite = (
        merged.groupby("month", as_index=False)
        .agg(
            regularity_mean=("regularity_pct", "mean"),
            cancellation_mean=("cancellation_pct", "mean"),
        )
        .sort_values("month")
    )
    month_stress = (
        merged.dropna(subset=["weather_stress_score"])
        .groupby("month", as_index=False)
        .agg(weather_stress_mean=("weather_stress_score", "mean"))
        .sort_values("month")
    )
    month_regularite["month_label"] = month_regularite["month"].map(MONTH_LABELS)
    month_stress["month_label"] = month_stress["month"].map(MONTH_LABELS)
    return {"regularity": month_regularite, "stress": month_stress}


def build_region_overview(merged: pd.DataFrame) -> pd.DataFrame:
    overview = (
        merged.groupby("region_current", as_index=False)
        .agg(
            regularite_moyenne=("regularity_pct", "mean"),
            annulation_moyenne=("cancellation_pct", "mean"),
            trains_programmes=("nombre_de_trains_programmes", "sum"),
        )
        .sort_values("regularite_moyenne")
        .reset_index(drop=True)
    )
    overview["display_region"] = overview["region_current"].map(display_region)
    return overview


def build_mode_summary(mode_monthly: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    shock_rows = []
    for mode, group in mode_monthly.groupby("mode"):
        fit = linear_fit(group["weather_stress_score"], group["performance_gap"])
        q20 = group["weather_stress_score"].quantile(0.20)
        q80 = group["weather_stress_score"].quantile(0.80)
        calm_gap = group.loc[group["weather_stress_score"] <= q20, "performance_gap"].mean()
        shock_gap = group.loc[group["weather_stress_score"] >= q80, "performance_gap"].mean()
        rows.append(
            {
                "mode": mode,
                "mode_display": display_mode(mode),
                "metric_label": group["metric_label"].iloc[0],
                "correlation": fit["correlation"],
                "slope": fit["slope"],
                "performance_moyenne": group["performance_pct"].mean(),
                "observations": len(group),
            }
        )
        shock_rows.append(
            {
                "mode": mode,
                "mode_display": display_mode(mode),
                "calm_gap_mean": calm_gap,
                "shock_gap_mean": shock_gap,
                "shock_minus_calm": shock_gap - calm_gap,
            }
        )
    return (
        pd.DataFrame(rows).sort_values("correlation").reset_index(drop=True),
        pd.DataFrame(shock_rows).sort_values("shock_minus_calm").reset_index(drop=True),
    )


def summarize_results(merged: pd.DataFrame, mode_monthly: pd.DataFrame, region_context: pd.DataFrame) -> dict:
    relationship = linear_fit(merged["weather_stress_score"], merged["regularity_gap"])
    cancellation_relationship = linear_fit(merged["weather_stress_score"], merged["cancellation_gap"])

    region_sensitivity_rows: List[dict] = []
    for region, group in merged.groupby("region_current"):
        subset = group.dropna(subset=["weather_stress_score", "regularity_gap"])
        if len(subset) < 36:
            continue
        fit = linear_fit(subset["weather_stress_score"], subset["regularity_gap"])
        region_sensitivity_rows.append(
            {
                "region_current": region,
                "display_region": display_region(region),
                "slope_regularite": fit["slope"],
                "correlation": fit["correlation"],
                "observations": len(subset),
            }
        )
    region_sensitivity = pd.DataFrame(region_sensitivity_rows).sort_values("slope_regularite").reset_index(drop=True)

    headline_metrics = pd.DataFrame(
        [
            {"indicateur": "Correlation stress meteo / ecart de regularite TER", "valeur": relationship["correlation"]},
            {"indicateur": "Effet d'une unite de stress sur la regularite TER", "valeur": relationship["slope"]},
            {"indicateur": "Correlation stress meteo / ecart d'annulation TER", "valeur": cancellation_relationship["correlation"]},
            {"indicateur": "Effet d'une unite de stress sur l'annulation TER", "valeur": cancellation_relationship["slope"]},
        ]
    )

    bucket_summary = (
        merged.groupby("stress_bucket", observed=False)
        .agg(
            observations=("date", "size"),
            regularity_gap_mean=("regularity_gap", "mean"),
            cancellation_gap_mean=("cancellation_gap", "mean"),
            regularity_pct_mean=("regularity_pct", "mean"),
            cancellation_pct_mean=("cancellation_pct", "mean"),
        )
        .reset_index()
    )

    top_weather_episodes = (
        merged[(merged["weather_stress_score"] >= merged["weather_stress_score"].quantile(0.85)) & (merged["regularity_gap"] < 0)]
        .sort_values(["regularity_gap", "weather_stress_score"], ascending=[True, False])
        .head(10)
        .copy()
    )
    top_weather_episodes["label"] = top_weather_episodes.apply(
        lambda row: f"{display_region(row['region_current'])} - {row['date']:%Y-%m}",
        axis=1,
    )
    top_weather_episodes["comment_clean"] = top_weather_episodes["commentaires"].map(shorten_comment)

    top_non_weather_episodes = (
        merged[merged["weather_stress_score"] <= merged["weather_stress_score"].quantile(0.20)]
        .sort_values("regularity_gap")
        .head(5)
        .copy()
    )

    mode_summary, mode_shocks = build_mode_summary(mode_monthly)
    context_sensitivity = region_sensitivity.merge(region_context, on="region_current", how="left")

    return {
        "headline_metrics": headline_metrics,
        "bucket_summary": bucket_summary,
        "region_sensitivity": region_sensitivity,
        "top_weather_episodes": top_weather_episodes,
        "top_non_weather_episodes": top_non_weather_episodes,
        "mode_summary": mode_summary,
        "mode_shocks": mode_shocks,
        "context_sensitivity": context_sensitivity,
    }


national = build_national_monthly(merged)
month_profiles = build_month_profiles(merged)
region_overview = build_region_overview(merged)
summary = summarize_results(merged, mode_monthly, region_context)
context_display = region_context.copy()
context_display["display_region"] = context_display["region_current"].map(display_region)

summary["headline_metrics"]
        


,indicateur,valeur
0,Correlation stress meteo / ecart de regularite...,-0.218698
1,Effet d'une unite de stress sur la regularite TER,-1.509139
2,Correlation stress meteo / ecart d'annulation TER,0.158003
3,Effet d'une unite de stress sur l'annulation TER,0.679163


In [140]:
summary["mode_summary"]
        


,mode,mode_display,metric_label,correlation,slope,performance_moyenne,observations
0,Intercites,Intercités,R?gularit? composite,-0.309341,-5.650783,84.840722,139
1,TER,TER,R?gularit? ? 5 minutes,-0.159093,-1.097308,91.311668,158
2,Transilien,Transilien,Ponctualit? voyageurs,-0.015817,-0.166682,89.643227,153


In [141]:
summary["region_sensitivity"][["display_region", "slope_regularite", "correlation", "observations"]]
        


,display_region,slope_regularite,correlation,observations
0,Nouvelle-Aquitaine,-2.255613,-0.276606,158
1,Normandie,-1.926113,-0.261276,158
2,Occitanie,-1.838416,-0.225375,158
3,Hauts-de-France,-1.333242,-0.195247,156
4,Centre-Val de Loire,-1.294669,-0.180851,158
5,Bretagne,-1.266760,-0.224274,158
6,Grand Est,-1.018226,-0.237262,123
7,Pays de la Loire,-0.952598,-0.154803,158


## Figures
        


In [142]:
def set_plot_style() -> None:
    plt.style.use("default")
    plt.rcParams.update(
        {
            "figure.facecolor": "white",
            "axes.facecolor": "white",
            "axes.edgecolor": "#B6C2CF",
            "axes.labelcolor": COLORS["text"],
            "axes.titlesize": 14,
            "axes.titleweight": "bold",
            "axes.grid": True,
            "grid.color": COLORS["grid"],
            "grid.alpha": 0.6,
            "grid.linewidth": 0.8,
            "xtick.color": COLORS["text"],
            "ytick.color": COLORS["text"],
            "font.size": 11,
        }
    )


set_plot_style()
        


In [143]:
def plot_national_regularity(national: pd.DataFrame, path: Path) -> None:
    fig, ax = plt.subplots(figsize=(14, 6.2))
    ax.plot(national["date"], national["regularity_pct"], color=COLORS["regularity"], linewidth=2.2, label="Regularite")
    ax.plot(national["date"], national["regularity_rolling_12m"], color=COLORS["accent"], linewidth=1.6, linestyle="--", label="Tendance 12 mois")
    ax.set_title("Regularite TER nationale")
    ax.set_xlabel("Date")
    ax.set_ylabel("Regularite (%)")
    ax.legend(frameon=False, loc="lower left")
    ax.xaxis.set_major_locator(mdates.YearLocator(base=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    add_source(fig, "Sources : SNCF Open Data")
    save_figure(fig, path)


figure_01_path = FIGURES_DIR / "01_regularite_ter_nationale.png"
plot_national_regularity(national, figure_01_path)
print(figure_01_path)
        


c:\Users\antoc\analyse-impact-meteo-regularite-sncf\figures\01_regularite_ter_nationale.png


![Figure 01](figures/01_regularite_ter_nationale.png)
        


In [144]:
def plot_national_stress(national: pd.DataFrame, path: Path) -> None:
    fig, ax = plt.subplots(figsize=(14, 6.2))
    ax.fill_between(national["date"], national["weather_stress_score"], color=COLORS["stress"], alpha=0.22)
    ax.plot(national["date"], national["weather_stress_score"], color=COLORS["stress"], linewidth=1.8, label="Score meteo")
    ax.plot(national["date"], national["stress_rolling_12m"], color=COLORS["accent"], linewidth=1.6, linestyle="--", label="Tendance 12 mois")
    ax.set_title("Stress meteo national")
    ax.set_xlabel("Date")
    ax.set_ylabel("Score meteo")
    ax.legend(frameon=False, loc="upper left")
    ax.xaxis.set_major_locator(mdates.YearLocator(base=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    add_source(fig, "Sources : Open-Meteo")
    save_figure(fig, path)


figure_02_path = FIGURES_DIR / "02_stress_meteo_national.png"
plot_national_stress(national, figure_02_path)
print(figure_02_path)
        


c:\Users\antoc\analyse-impact-meteo-regularite-sncf\figures\02_stress_meteo_national.png


![Figure 02](figures/02_stress_meteo_national.png)
        


In [145]:
def plot_month_regularity_profile(month_regularite: pd.DataFrame, path: Path) -> None:
    fig, ax = plt.subplots(figsize=(10.5, 5.8))
    ax.bar(
        month_regularite["month_label"],
        month_regularite["regularity_mean"],
        color=COLORS["regularity"],
        edgecolor="white",
        linewidth=0.8,
    )
    ax.set_title("Regularite moyenne par mois")
    ax.set_xlabel("Mois")
    ax.set_ylabel("Regularite moyenne (%)")
    add_value_labels(ax, padding=0.05)
    add_source(fig, "Sources : SNCF Open Data")
    save_figure(fig, path)


figure_03_path = FIGURES_DIR / "03_regularite_moyenne_par_mois.png"
plot_month_regularity_profile(month_profiles["regularity"], figure_03_path)
print(figure_03_path)
            


c:\Users\antoc\analyse-impact-meteo-regularite-sncf\figures\03_regularite_moyenne_par_mois.png


![Figure 03](figures/03_regularite_moyenne_par_mois.png)
        


In [146]:
def plot_month_stress_profile(month_stress: pd.DataFrame, path: Path) -> None:
    fig, ax = plt.subplots(figsize=(10.5, 5.8))
    ax.bar(
        month_stress["month_label"],
        month_stress["weather_stress_mean"],
        color=COLORS["stress"],
        edgecolor="white",
        linewidth=0.8,
    )
    ax.set_title("Stress meteo moyen par mois")
    ax.set_xlabel("Mois")
    ax.set_ylabel("Score meteo moyen")
    add_value_labels(ax, padding=0.01)
    add_source(fig, "Sources : Open-Meteo")
    save_figure(fig, path)


figure_04_path = FIGURES_DIR / "04_stress_meteo_moyen_par_mois.png"
plot_month_stress_profile(month_profiles["stress"], figure_04_path)
print(figure_04_path)
            


c:\Users\antoc\analyse-impact-meteo-regularite-sncf\figures\04_stress_meteo_moyen_par_mois.png


![Figure 04](figures/04_stress_meteo_moyen_par_mois.png)
        


In [147]:
def plot_scatter_relationship(merged: pd.DataFrame, summary: dict, path: Path) -> None:
    fig, ax = plt.subplots(figsize=(12.5, 7.2))
    sizes = 30 + 180 * (
        merged["nombre_de_trains_programmes"] / merged["nombre_de_trains_programmes"].quantile(0.95)
    ).clip(upper=1.5)
    scatter = ax.scatter(
        merged["weather_stress_score"],
        merged["regularity_gap"],
        s=sizes,
        c=merged["weather_stress_score"],
        cmap="YlOrRd",
        alpha=0.65,
        edgecolors="white",
        linewidths=0.4,
    )
    fit = linear_fit(merged["weather_stress_score"], merged["regularity_gap"])
    x_line = np.linspace(merged["weather_stress_score"].min(), merged["weather_stress_score"].max(), 200)
    y_line = fit["intercept"] + fit["slope"] * x_line
    ax.plot(x_line, y_line, color=COLORS["accent"], linewidth=2.2)
    ax.axhline(0, color="#7A869A", linestyle="--", linewidth=1)
    ax.set_title("Stress meteo et ecart de regularite")
    ax.set_xlabel("Score meteo mensuel")
    ax.set_ylabel("Ecart a la regularite habituelle (points)")
    cbar = fig.colorbar(scatter, ax=ax)
    cbar.set_label("Intensite du stress")
    corr_value = summary["headline_metrics"].iloc[0, 1]
    slope_value = summary["headline_metrics"].iloc[1, 1]
    ax.text(
        0.02,
        0.98,
        f"Correlation = {corr_value:.2f}\nPente = {slope_value:.2f}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        bbox={"boxstyle": "round,pad=0.4", "facecolor": "white", "edgecolor": COLORS["grid"]},
    )
    add_source(fig, "Sources : SNCF Open Data, Open-Meteo")
    save_figure(fig, path)


figure_05_path = FIGURES_DIR / "05_relation_stress_regularite.png"
plot_scatter_relationship(merged, summary, figure_05_path)
print(figure_05_path)
        


c:\Users\antoc\analyse-impact-meteo-regularite-sncf\figures\05_relation_stress_regularite.png


![Figure 05](figures/05_relation_stress_regularite.png)
        


In [148]:
def plot_shock_vs_calm(bucket_summary: pd.DataFrame, path: Path) -> None:
    ordered = bucket_summary.set_index("stress_bucket").reindex(STRESS_BUCKET_ORDER).reset_index()
    numeric_cols = [
        "observations",
        "regularity_gap_mean",
        "cancellation_gap_mean",
        "regularity_pct_mean",
        "cancellation_pct_mean",
    ]
    ordered[numeric_cols] = ordered[numeric_cols].fillna(0)
    palette = {
        "Mois calmes": COLORS["calm"],
        "Mois intermediaires": COLORS["middle"],
        "Mois chocs meteo": COLORS["shock"],
    }
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.4), sharex=True)
    axes[0].bar(
        ordered["stress_bucket"],
        ordered["regularity_gap_mean"],
        color=[palette[label] for label in ordered["stress_bucket"]],
    )
    axes[0].axhline(0, color="#7A869A", linestyle="--", linewidth=1)
    axes[0].set_title("Ecart moyen de regularite")
    axes[0].set_ylabel("Points")
    add_value_labels(axes[0], padding=0.04)
    axes[1].bar(
        ordered["stress_bucket"],
        ordered["cancellation_gap_mean"],
        color=[palette[label] for label in ordered["stress_bucket"]],
    )
    axes[1].axhline(0, color="#7A869A", linestyle="--", linewidth=1)
    axes[1].set_title("Ecart moyen d'annulation")
    axes[1].set_ylabel("Points")
    add_value_labels(axes[1], padding=0.04)
    for ax in axes:
        ax.tick_params(axis="x", rotation=15)
    add_source(fig, "Sources : SNCF Open Data, Open-Meteo")
    save_figure(fig, path)


figure_06_path = FIGURES_DIR / "06_mois_calmes_vs_mois_chocs.png"
plot_shock_vs_calm(summary["bucket_summary"], figure_06_path)
print(figure_06_path)
        


c:\Users\antoc\analyse-impact-meteo-regularite-sncf\figures\06_mois_calmes_vs_mois_chocs.png


![Figure 06](figures/06_mois_calmes_vs_mois_chocs.png)
        


In [149]:
def plot_region_regularite_bar(region_overview: pd.DataFrame, path: Path) -> None:
    ordered = region_overview.sort_values("regularite_moyenne")
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.barh(
        ordered["display_region"],
        ordered["regularite_moyenne"],
        color=COLORS["regularity"],
        edgecolor="white",
        linewidth=0.8,
    )
    ax.set_title("Regularite TER moyenne par region")
    ax.set_xlabel("Regularite moyenne (%)")
    add_horizontal_labels(ax, fmt="{:.1f}", offset=0.12)
    add_source(fig, "Sources : SNCF Open Data")
    save_figure(fig, path)


figure_07_path = FIGURES_DIR / "07_regularite_moyenne_par_region.png"
plot_region_regularite_bar(region_overview, figure_07_path)
print(figure_07_path)
            


c:\Users\antoc\analyse-impact-meteo-regularite-sncf\figures\07_regularite_moyenne_par_region.png


![Figure 07](figures/07_regularite_moyenne_par_region.png)
        


In [150]:
def fetch_regions_geojson(force_refresh: bool = False) -> dict:
    if REGIONS_GEOJSON_PATH.exists() and not force_refresh:
        return json.loads(REGIONS_GEOJSON_PATH.read_text(encoding="utf-8"))
    response = requests.get(REGIONS_GEOJSON_URL, timeout=120)
    response.raise_for_status()
    REGIONS_GEOJSON_PATH.write_text(response.text, encoding="utf-8")
    return response.json()


def plot_france_region_heatmap(region_overview: pd.DataFrame, mode_monthly: pd.DataFrame, path: Path) -> None:
    geojson = fetch_regions_geojson()
    values = {
        display_region(row.region_current): row.regularite_moyenne
        for row in region_overview.itertuples(index=False)
    }

    transilien = mode_monthly.loc[
        mode_monthly["mode"].map(canonical_mode) == "Transilien",
        "performance_pct",
    ].dropna()
    if not transilien.empty:
        values["Île-de-France"] = float(transilien.mean())

    patches = []
    patch_values = []
    labels = []
    min_x, min_y = float("inf"), float("inf")
    max_x, max_y = float("-inf"), float("-inf")

    for feature in geojson["features"]:
        region_name = feature["properties"]["nom"]
        canonical = canonical_region(region_name)
        if canonical not in MAP_LABELS:
            continue
        if region_name not in values:
            continue

        geometry = shapely_shape(feature["geometry"])
        bounds = geometry.bounds
        min_x = min(min_x, bounds[0])
        min_y = min(min_y, bounds[1])
        max_x = max(max_x, bounds[2])
        max_y = max(max_y, bounds[3])

        polygons = list(geometry.geoms) if geometry.geom_type == "MultiPolygon" else [geometry]
        for polygon in polygons:
            coords = np.asarray(polygon.exterior.coords)
            patches.append(MplPolygon(coords[:, :2], closed=True))
            patch_values.append(values[region_name])

        point = geometry.representative_point()
        label = MAP_LABELS.get(canonical, region_name)
        if canonical == "Ile-de-France":
            label = f"{label}*"
        labels.append((point.x, point.y, label))

    fig, ax = plt.subplots(figsize=(10.2, 8.0))
    collection = PatchCollection(patches, cmap="YlOrRd", edgecolor="white", linewidth=1.0)
    collection.set_array(np.asarray(patch_values))
    ax.add_collection(collection)
    ax.autoscale_view()
    ax.set_aspect("equal")
    ax.set_xlim(min_x - 0.8, max_x + 0.8)
    ax.set_ylim(min_y - 0.5, max_y + 0.5)
    ax.axis("off")
    ax.set_title("Carte regionale de la regularite")

    for x, y, label in labels:
        ax.text(x, y, label, ha="center", va="center", fontsize=9, color="#1F2937")

    cbar = fig.colorbar(collection, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label("Regularite moyenne (%)")
    add_source(fig, "Sources : SNCF Open Data, data.gouv.fr")
    fig.text(0.99, 0.01, "* donnees Transilien", ha="right", fontsize=9, color="#5E6C84")
    save_figure(fig, path)


figure_08_path = FIGURES_DIR / "08_carte_france_regularite_ter.png"
plot_france_region_heatmap(region_overview, mode_monthly, figure_08_path)
print(figure_08_path)
            


c:\Users\antoc\analyse-impact-meteo-regularite-sncf\figures\08_carte_france_regularite_ter.png


![Figure 08](figures/08_carte_france_regularite_ter.png)
        


In [151]:
def plot_region_sensitivity(region_sensitivity: pd.DataFrame, path: Path) -> None:
    ordered = region_sensitivity.sort_values("slope_regularite").copy()
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.barh(
        ordered["display_region"],
        ordered["slope_regularite"],
        color=np.where(ordered["slope_regularite"] < ordered["slope_regularite"].median(), COLORS["shock"], COLORS["regularity"]),
        alpha=0.9,
    )
    ax.axvline(0, color="#7A869A", linestyle="--", linewidth=1)
    ax.set_title("Sensibilite meteo par region")
    ax.set_xlabel("Perte estimee de regularite par unite de stress")
    add_source(fig, "Sources : SNCF Open Data, Open-Meteo")
    save_figure(fig, path)


figure_09_path = FIGURES_DIR / "09_sensibilite_regionale_a_la_meteo.png"
plot_region_sensitivity(summary["region_sensitivity"], figure_09_path)
print(figure_09_path)
        


c:\Users\antoc\analyse-impact-meteo-regularite-sncf\figures\09_sensibilite_regionale_a_la_meteo.png


![Figure 09](figures/09_sensibilite_regionale_a_la_meteo.png)
        


In [152]:
def plot_mode_average_levels(mode_summary: pd.DataFrame, path: Path) -> None:
    averages = mode_summary.sort_values("performance_moyenne").copy()
    fig, ax = plt.subplots(figsize=(8.8, 5.8))
    ax.bar(
        averages["mode_display"],
        averages["performance_moyenne"],
        color=[MODE_COLORS[mode] for mode in averages["mode"]],
        edgecolor="white",
        linewidth=0.8,
    )
    ax.set_title("Performance moyenne par mode")
    ax.set_ylabel("Performance moyenne (%)")
    add_value_labels(ax, padding=0.08)
    add_source(fig, "Sources : SNCF Open Data")
    save_figure(fig, path)


figure_10_path = FIGURES_DIR / "10_niveau_moyen_par_mode.png"
plot_mode_average_levels(summary["mode_summary"], figure_10_path)
print(figure_10_path)
        


c:\Users\antoc\analyse-impact-meteo-regularite-sncf\figures\10_niveau_moyen_par_mode.png


![Figure 10](figures/10_niveau_moyen_par_mode.png)
        


In [153]:
def plot_mode_comparison(mode_summary: pd.DataFrame, mode_shocks: pd.DataFrame, path: Path) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.6))
    ordered_corr = mode_summary.sort_values("correlation")
    axes[0].bar(
        ordered_corr["mode_display"],
        ordered_corr["correlation"],
        color=[MODE_COLORS[mode] for mode in ordered_corr["mode"]],
        alpha=0.9,
    )
    axes[0].axhline(0, color="#7A869A", linestyle="--", linewidth=1)
    axes[0].set_title("Correlation meteo / performance")
    axes[0].set_ylabel("Correlation")
    add_value_labels(axes[0], padding=0.01)
    ordered_shock = mode_shocks.sort_values("shock_minus_calm")
    axes[1].bar(
        ordered_shock["mode_display"],
        ordered_shock["shock_minus_calm"],
        color=[MODE_COLORS[mode] for mode in ordered_shock["mode"]],
        alpha=0.9,
    )
    axes[1].axhline(0, color="#7A869A", linestyle="--", linewidth=1)
    axes[1].set_title("Ecart mois calmes / mois chocs")
    axes[1].set_ylabel("Points de performance")
    add_value_labels(axes[1], padding=0.03)
    add_source(fig, "Sources : SNCF Open Data, Open-Meteo")
    save_figure(fig, path)


figure_11_path = FIGURES_DIR / "11_comparaison_modes_meteo.png"
plot_mode_comparison(summary["mode_summary"], summary["mode_shocks"], figure_11_path)
print(figure_11_path)
        


c:\Users\antoc\analyse-impact-meteo-regularite-sncf\figures\11_comparaison_modes_meteo.png


![Figure 11](figures/11_comparaison_modes_meteo.png)
        


In [154]:
def plot_context_vs_sensitivity(context_sensitivity: pd.DataFrame, path: Path) -> None:
    fig, ax = plt.subplots(figsize=(12.5, 7.4))
    sizes = 60 + 260 * (
        context_sensitivity["frequentation_voyageurs_2024"] / context_sensitivity["frequentation_voyageurs_2024"].max()
    ).fillna(0)
    scatter = ax.scatter(
        context_sensitivity["gare_count"],
        context_sensitivity["slope_regularite"],
        s=sizes,
        c=context_sensitivity["voyageurs_par_gare_2024"],
        cmap="YlGnBu",
        edgecolors="white",
        linewidths=0.7,
        alpha=0.95,
    )
    ax.axhline(0, color="#7A869A", linestyle="--", linewidth=1)
    ax.set_title("Contexte reseau et sensibilite meteo")
    ax.set_xlabel("Nombre de gares voyageurs")
    ax.set_ylabel("Sensibilite meteo estimee")
    for row in context_sensitivity.itertuples(index=False):
        ax.text(row.gare_count + 1.2, row.slope_regularite, row.display_region, fontsize=9, color=COLORS["text"])
    cbar = fig.colorbar(scatter, ax=ax)
    cbar.set_label("Voyageurs 2024 par gare")
    add_source(fig, "Sources : SNCF Open Data, Gares & Connexions, Open-Meteo")
    save_figure(fig, path)


figure_12_path = FIGURES_DIR / "12_contexte_reseau_et_sensibilite.png"
plot_context_vs_sensitivity(summary["context_sensitivity"], figure_12_path)
print(figure_12_path)
        


c:\Users\antoc\analyse-impact-meteo-regularite-sncf\figures\12_contexte_reseau_et_sensibilite.png


![Figure 12](figures/12_contexte_reseau_et_sensibilite.png)
        


In [155]:
def plot_top_weather_episodes(top_weather_episodes: pd.DataFrame, path: Path) -> None:
    ordered = top_weather_episodes.sort_values("regularity_gap").copy()
    fig, ax = plt.subplots(figsize=(12.8, 6.8))
    ax.barh(
        ordered["label"],
        ordered["regularity_gap"],
        color=COLORS["shock"],
        edgecolor="white",
        linewidth=0.8,
    )
    ax.axvline(0, color="#7A869A", linestyle="--", linewidth=1)
    ax.set_title("Principaux episodes meteo")
    ax.set_xlabel("Ecart a la regularite habituelle (points)")
    add_source(fig, "Sources : SNCF Open Data, Open-Meteo")
    save_figure(fig, path)


figure_13_path = FIGURES_DIR / "13_episodes_meteo_marquants.png"
plot_top_weather_episodes(summary["top_weather_episodes"], figure_13_path)
print(figure_13_path)
            


c:\Users\antoc\analyse-impact-meteo-regularite-sncf\figures\13_episodes_meteo_marquants.png


![Figure 13](figures/13_episodes_meteo_marquants.png)
        


In [156]:
def plot_network_context(region_context: pd.DataFrame, path: Path) -> None:
    ordered = region_context.sort_values("gare_count").copy()
    ordered["display_region"] = ordered["region_current"].map(display_region)
    fig, axes = plt.subplots(1, 2, figsize=(15, 7), sharey=True)
    axes[0].barh(
        ordered["display_region"],
        ordered["gare_count"],
        color=COLORS["accent"],
        edgecolor="white",
        linewidth=0.8,
    )
    axes[0].set_title("Nombre de gares par region")
    axes[0].set_xlabel("Gares")
    axes[1].barh(
        ordered["display_region"],
        ordered["frequentation_voyageurs_2024"] / 1_000_000,
        color=COLORS["stress"],
        edgecolor="white",
        linewidth=0.8,
    )
    axes[1].set_title("Frequentation 2024 par region")
    axes[1].set_xlabel("Millions de voyageurs")
    add_source(fig, "Sources : Gares & Connexions, SNCF Open Data")
    save_figure(fig, path)


figure_14_path = FIGURES_DIR / "14_gares_et_frequentation_par_region.png"
plot_network_context(region_context, figure_14_path)
print(figure_14_path)
            


c:\Users\antoc\analyse-impact-meteo-regularite-sncf\figures\14_gares_et_frequentation_par_region.png


![Figure 14](figures/14_gares_et_frequentation_par_region.png)
        


In [157]:
def plot_month_cancellation_profile(month_regularite: pd.DataFrame, path: Path) -> None:
    fig, ax = plt.subplots(figsize=(10.5, 5.8))
    ax.bar(
        month_regularite["month_label"],
        month_regularite["cancellation_mean"],
        color=COLORS["cancellation"],
        edgecolor="white",
        linewidth=0.8,
    )
    ax.set_title("Annulation moyenne par mois")
    ax.set_xlabel("Mois")
    ax.set_ylabel("Annulation moyenne (%)")
    add_value_labels(ax, padding=0.02)
    add_source(fig, "Sources : SNCF Open Data")
    save_figure(fig, path)


figure_15_path = FIGURES_DIR / "15_annulation_moyenne_par_mois.png"
plot_month_cancellation_profile(month_profiles["regularity"], figure_15_path)
print(figure_15_path)
            


c:\Users\antoc\analyse-impact-meteo-regularite-sncf\figures\15_annulation_moyenne_par_mois.png


![Figure 15](figures/15_annulation_moyenne_par_mois.png)
            


In [158]:
def plot_mode_timeline(mode_monthly: pd.DataFrame, path: Path) -> None:
    fig, ax = plt.subplots(figsize=(14, 6.2))
    ordered = mode_monthly.copy()
    ordered["mode"] = ordered["mode"].map(canonical_mode)
    ordered = ordered.sort_values("date")

    for mode in ["TER", "Intercites", "Transilien"]:
        subset = ordered.loc[ordered["mode"] == mode, ["date", "performance_pct"]].copy()
        if subset.empty:
            continue
        subset["trend_6m"] = subset["performance_pct"].rolling(6, min_periods=2).mean()
        ax.plot(
            subset["date"],
            subset["trend_6m"],
            linewidth=2.4,
            color=MODE_COLORS[mode],
            label=display_mode(mode),
        )

    ax.set_title("Performance dans le temps par mode")
    ax.set_xlabel("Date")
    ax.set_ylabel("Performance (%)")
    ax.legend(frameon=False, loc="lower left")
    ax.xaxis.set_major_locator(mdates.YearLocator(base=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    add_source(fig, "Sources : SNCF Open Data")
    save_figure(fig, path)


figure_16_path = FIGURES_DIR / "16_performance_dans_le_temps_par_mode.png"
plot_mode_timeline(mode_monthly, figure_16_path)
print(figure_16_path)
            


c:\Users\antoc\analyse-impact-meteo-regularite-sncf\figures\16_performance_dans_le_temps_par_mode.png


![Figure 16](figures/16_performance_dans_le_temps_par_mode.png)
            


## Tables
        


In [159]:
summary["top_weather_episodes"][["label", "weather_stress_score", "regularity_gap", "cancellation_pct", "comment_clean"]]
        


,label,weather_stress_score,regularity_gap,cancellation_pct,comment_clean
1295,Nouvelle-Aquitaine - 2023-11,1.054121,-9.315758,5.387292,Pas de commentaire SNCF disponible.
872,Nouvelle-Aquitaine - 2019-12,0.573145,-8.952900,6.507202,Pas de commentaire SNCF disponible.
1539,Occitanie - 2026-02,0.807809,-7.534338,10.962151,Pas de commentaire SNCF disponible.
24,Bretagne - 2013-03,1.134169,-6.979326,1.853402,Perte de régularité supérieure à 6 points en g...
670,Centre-Val de Loire - 2018-02,0.613225,-6.255253,2.788442,Un mois de février catastrophique en matière d...
889,Normandie - 2020-02,1.118639,-6.194228,3.595874,Pas de commentaire SNCF disponible.
1540,Pays de la Loire - 2026-02,0.774474,-5.566012,1.658661,Pas de commentaire SNCF disponible.
961,Normandie - 2020-10,0.528366,-5.423510,2.065374,Pas de commentaire SNCF disponible.
1512,Occitanie - 2025-11,1.364624,-5.103135,4.963222,Pas de commentaire SNCF disponible.
661,Centre-Val de Loire - 2018-01,0.714654,-5.101128,2.041680,Ce premier mois de l'année amorce une dynamiqu...


In [160]:
summary["top_non_weather_episodes"][["date", "region_current", "regularity_gap", "weather_stress_score", "commentaires"]]
        


,date,region_current,regularity_gap,weather_stress_score,commentaires
880,2020-01-01,Normandie,-8.288452,0.0,NaN
563,2017-04-01,Bretagne,-6.841293,0.0,NaN
756,2018-11-01,Occitanie,-3.943637,0.0,NaN
436,2016-04-01,Nouvelle-Aquitaine,-3.076839,0.0,"Les travaux en gare de Bordeaux, mais égalemen..."
647,2017-11-01,Pays de la Loire,-2.786784,0.0,NaN


Notre ordre de lecture est le suivant : vue nationale, saisonnalite, annulations, relation meteo / regularite, lecture regionale, comparaison des modes, puis contexte reseau.
                
